In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Crear conexión a base de datos en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Crear tablas de ejemplo (Customers y Sales)
cursor.execute('''
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT,
    country TEXT,
    join_date DATE
)''')

cursor.execute('''
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_name TEXT,
    amount REAL,
    sale_date DATE
)''')

# Insertar datos de prueba
customers_data = [
    (1, 'Alice Jones', 'USA', '2023-01-15'),
    (2, 'Bob Smith', 'UK', '2023-02-10'),
    (3, 'Charlie Brown', 'USA', '2023-03-05'),
    (4, 'David Garcia', 'Spain', '2023-01-20'),
    (5, 'Eve Müller', 'Germany', '2023-05-12')
]

sales_data = [
    (101, 1, 'Laptop', 1200.0, '2024-01-10'),
    (102, 1, 'Mouse', 25.0, '2024-02-15'),
    (103, 2, 'Smartphone', 800.0, '2024-01-20'),
    (104, 3, 'Tablet', 450.0, '2024-01-05'),
    (105, 4, 'Monitor', 300.0, '2023-11-10'),
    (106, 1, 'Headphones', 150.0, '2024-03-01'),
    (107, 2, 'Charger', 20.0, '2024-01-25')
]

cursor.executemany('INSERT INTO customers VALUES (?,?,?,?)', customers_data)
cursor.executemany('INSERT INTO sales VALUES (?,?,?,?,?)', sales_data)
conn.commit()

print("✅ Base de datos SQL lista con tablas 'customers' y 'sales'.")

## 🔍 Key Queries: Análisis de Clientes Top
Calcularemos quiénes son los clientes que más han gastado en la plataforma.

In [ ]:
query_top_customers = """
SELECT c.name, c.country, SUM(s.amount) as total_spent
FROM customers c
JOIN sales s ON c.id = s.customer_id
GROUP BY c.id
ORDER BY total_spent DESC;
"""

df_top = pd.read_sql_query(query_top_customers, conn)
df_top

## 🌍 Revenue by Country
Análisis de ingresos totales distribuidos por región geográfica.

In [ ]:
query_revenue_country = """
SELECT c.country, SUM(s.amount) as revenue
FROM customers c
JOIN sales s ON c.id = s.customer_id
GROUP BY c.country
ORDER BY revenue DESC;
"""

df_country = pd.read_sql_query(query_revenue_country, conn)

# Visualización rápida
plt.figure(figsize=(8, 4))
sns.barplot(data=df_country, x='country', y='revenue', palette='viridis')
plt.title('Ingresos Totales por País')
plt.show()

## ⚠️ Churn Detection
Identificaremos clientes que no han realizado compras en los últimos meses (considerando como "reciente" el año 2024).

In [ ]:
query_churn = """
SELECT name, country 
FROM customers 
WHERE id NOT IN (
    SELECT DISTINCT customer_id 
    FROM sales 
    WHERE sale_date >= '2024-01-01'
);
"""

df_churn = pd.read_sql_query(query_churn, conn)
print("Clientes en riesgo de abandono (Inactivos en 2024):")
df_churn

## 💡 Business Impact & Conclusions
- **Retención:** Se han identificado segmentos inactivos (ej. David Garcia) para campañas de re-engagement.
- **Estrategia:** USA lidera el mercado en volumen de ventas, lo que sugiere optimizar logística en esa zona.
- **Oportunidad:** Una minoría de clientes (Alice Jones) representa la mayoría de las compras de alto ticket.